In [1]:
from os import rename

import pandas as pd
import time
import json
import importlib
import config
importlib.reload(config)

from config import US_TICKERS_FILE_01



In [73]:
with open(US_TICKERS_FILE_01, 'r') as f:
    tickers = json.load(f)

df_tickers = pd.DataFrame.from_dict(tickers, orient='index', columns=['Ticker', 'Start_Date', 'End_Date'])
df_tickers.index.name = 'ISIN'
df_tickers = df_tickers.reset_index()

print(df_tickers.shape)
print(df_tickers.head())

(28853, 4)
           ISIN Ticker  Start_Date    End_Date
0  US0970231058     BA  2018-01-01  2025-06-30
1  US0846701086   BRKA  2018-01-01  2025-06-30
2  US0846707026   BRKB  2018-01-01  2025-06-30
3  US12572Q1058    CME  2018-01-01  2025-06-30
4  US7081601061    JCP  2018-01-01  2021-01-06


In [85]:
df_tickers_us = df_tickers[df_tickers['ISIN'].str.startswith('US')]
print(df_tickers_us.shape)

(18328, 4)


In [74]:
df_tickers['Ticker'].nunique()

24272

In [75]:
df_tickers['ISIN'].nunique()

28853

In [5]:
PRICE_FILE_PATH = r"T:\CGATES\Alexandria\US_All_Universe_Returns_Daily_Sectors_2000_20240328.txt"
df_prices = pd.read_csv(PRICE_FILE_PATH, sep='\t')



In [31]:
df_prices['Ticker'].nunique()

3738

In [66]:
df_prices['Date'] = pd.to_datetime(df_prices['Date'])
dfp = df_prices[df_prices['Date']>='2018-01-01']
print(dfp.shape)
print(dfp['Ticker'].nunique())

(2358228, 6)
2084


In [67]:
dfp.head(50)

,Date,Ticker,D0,D1,Return,Sector
6798518,2018-01-02,A,67.60,69.32,2.54%,Information Technology
6798519,2018-01-02,AAL,52.99,52.34,-1.23%,Industrials
6798520,2018-01-02,AAN,39.47,39.67,0.51%,Consumer Discretionary
6798521,2018-01-02,AAOI,37.91,37.89,-0.05%,Information Technology
6798522,2018-01-02,AAON,36.95,37.60,1.76%,Industrials
6798523,2018-01-02,AAP,106.09,107.05,0.90%,Consumer Discretionary
6798524,2018-01-02,AAPL,172.26,172.23,-0.02%,Information Technology
6798525,2018-01-02,AAT,38.16,37.82,-0.89%,Financials
6798526,2018-01-02,AAWW,57.95,58.30,0.60%,Industrials
6798527,2018-01-02,AAXN,26.55,26.75,0.75%,Industrials


In [76]:
x = dfp[['Ticker', 'Sector']].drop_duplicates()

df = df_tickers.merge(x, on='Ticker', how='inner')

print(x.shape)
print(df_tickers.shape)

print(df.shape)
print(df.head(50))

(2400, 2)
(28853, 4)
(3065, 5)
            ISIN Ticker  Start_Date    End_Date                      Sector
0   US0970231058     BA  2018-01-01  2025-06-30                 Industrials
1   US0846707026   BRKB  2018-01-01  2025-06-30                  Financials
2   US12572Q1058    CME  2018-01-01  2025-06-30                  Financials
3   US7081601061    JCP  2018-01-01  2021-01-06      Consumer Discretionary
4   US55616P1049      M  2018-01-01  2025-06-07      Consumer Discretionary
5   US5949181045   MSFT  2018-01-01  2025-06-30      Information Technology
6   US5894331017    MDP  2018-01-01  2021-12-03      Consumer Discretionary
7   US5894331017    MDP  2018-01-01  2021-12-03  Telecommunication Services
8   US5894331017    MDP  2018-01-01  2021-12-03      Communication Services
9   US8872281048   TIME  2018-01-01  2018-01-31      Consumer Discretionary
10  US1380981084    CMD  2018-01-01  2021-06-02                 Health Care
11  US2166484020    COO  2018-01-01  2025-06-17          

In [77]:
print(df_tickers[df_tickers.duplicated(subset='Ticker', keep=False)].sort_values('Ticker').head(20))

               ISIN Ticker  Start_Date    End_Date
1771   US00846U1016      A  2018-01-03  2025-06-24
24988  CA04226J1084      A  2022-08-22  2022-10-04
16626  CA0120271089     AA  2020-01-28  2020-01-28
5021   US0138721065     AA  2018-01-13  2025-06-24
23369  CA33719F1099  AAA.P  2021-12-10  2025-06-24
11317  CA0534851088  AAA.P  2018-05-30  2021-08-27
23643  US38150K1034   AAAU  2022-01-19  2022-03-02
12213  US7154261025   AAAU  2018-08-15  2018-08-15
2183   US0003071083    AAC  2018-01-03  2021-07-24
22831  KYG330321061    AAC  2021-10-24  2023-11-01
18271  KY04316G1057   AACQ  2020-09-03  2021-06-25
17951  US04316G2049   AACQ  2020-07-17  2020-07-17
26017  KYG330331128   AACT  2023-04-26  2023-06-12
26369  KYG330331045   AACT  2023-08-21  2025-06-24
21736  US0240031056   AADI  2021-08-27  2021-09-14
21985  US00032Q1040   AADI  2021-09-20  2025-06-19
5476   CA00782P1080    AAL  2018-01-17  2020-04-14
1347   US02376R1023    AAL  2018-01-02  2025-06-30
27496  KYG1000R1193    AAM  202

In [78]:
dup_counts = df_tickers[df_tickers.duplicated(subset='Ticker', keep=False)].groupby('Ticker').size()
print(dup_counts.value_counts())

2    3207
3     552
4      80
5       7
Name: count, dtype: int64


In [79]:
dups = df_tickers[df_tickers.duplicated(subset='Ticker', keep=False)].sort_values(['Ticker','Start_Date'])
print(dups.head(20))

               ISIN Ticker  Start_Date    End_Date
1771   US00846U1016      A  2018-01-03  2025-06-24
24988  CA04226J1084      A  2022-08-22  2022-10-04
5021   US0138721065     AA  2018-01-13  2025-06-24
16626  CA0120271089     AA  2020-01-28  2020-01-28
11317  CA0534851088  AAA.P  2018-05-30  2021-08-27
23369  CA33719F1099  AAA.P  2021-12-10  2025-06-24
12213  US7154261025   AAAU  2018-08-15  2018-08-15
23643  US38150K1034   AAAU  2022-01-19  2022-03-02
2183   US0003071083    AAC  2018-01-03  2021-07-24
22831  KYG330321061    AAC  2021-10-24  2023-11-01
17951  US04316G2049   AACQ  2020-07-17  2020-07-17
18271  KY04316G1057   AACQ  2020-09-03  2021-06-25
26017  KYG330331128   AACT  2023-04-26  2023-06-12
26369  KYG330331045   AACT  2023-08-21  2025-06-24
21736  US0240031056   AADI  2021-08-27  2021-09-14
21985  US00032Q1040   AADI  2021-09-20  2025-06-19
1347   US02376R1023    AAL  2018-01-02  2025-06-30
5476   CA00782P1080    AAL  2018-01-17  2020-04-14
27496  KYG1000R1193    AAM  202

In [81]:
date_ranges = dfp.groupby('Ticker').agg(
    min_date=('Date', 'min'),
    max_date=('Date', 'max'),
    Sector=('Sector', 'first')
).reset_index()

print(date_ranges.shape)
print(date_ranges.head())

(2084, 4)
  Ticker   min_date   max_date                  Sector
0      A 2018-01-02 2024-03-28  Information Technology
1     AA 2019-01-02 2024-03-28               Materials
2    AAL 2018-01-02 2024-03-28             Industrials
3    AAN 2018-01-02 2024-01-31  Consumer Discretionary
4   AAOI 2018-01-02 2021-11-30  Information Technology


In [82]:
df = df_tickers.merge(date_ranges, on='Ticker', how='inner')

print(date_ranges.shape)
print(df_tickers.shape)

print(df.shape)
print(df.head(50))

(2084, 4)
(28853, 4)
(2655, 7)
            ISIN Ticker  Start_Date    End_Date   min_date   max_date  \
0   US0970231058     BA  2018-01-01  2025-06-30 2018-01-02 2024-03-28   
1   US0846707026   BRKB  2018-01-01  2025-06-30 2018-01-02 2024-03-28   
2   US12572Q1058    CME  2018-01-01  2025-06-30 2018-01-02 2024-03-28   
3   US7081601061    JCP  2018-01-01  2021-01-06 2018-01-02 2020-04-30   
4   US55616P1049      M  2018-01-01  2025-06-07 2018-01-02 2024-03-28   
5   US5949181045   MSFT  2018-01-01  2025-06-30 2018-01-02 2024-03-28   
6   US5894331017    MDP  2018-01-01  2021-12-03 2018-01-02 2021-11-30   
7   US8872281048   TIME  2018-01-01  2018-01-31 2018-01-02 2018-01-29   
8   US1380981084    CMD  2018-01-01  2021-06-02 2018-01-02 2021-05-28   
9   US2166484020    COO  2018-01-01  2025-06-17 2018-01-02 2024-03-28   
10  US2521311074   DXCM  2018-01-01  2025-06-30 2020-05-01 2024-03-28   
11  US28176E1082     EW  2018-01-01  2025-06-23 2018-01-02 2024-03-28   
12  US67018T1051    

In [83]:
dups = df_tickers[df_tickers.duplicated(subset='Ticker', keep=False)].sort_values(['Ticker','Start_Date'])
print(dups.head(20))

               ISIN Ticker  Start_Date    End_Date
1771   US00846U1016      A  2018-01-03  2025-06-24
24988  CA04226J1084      A  2022-08-22  2022-10-04
5021   US0138721065     AA  2018-01-13  2025-06-24
16626  CA0120271089     AA  2020-01-28  2020-01-28
11317  CA0534851088  AAA.P  2018-05-30  2021-08-27
23369  CA33719F1099  AAA.P  2021-12-10  2025-06-24
12213  US7154261025   AAAU  2018-08-15  2018-08-15
23643  US38150K1034   AAAU  2022-01-19  2022-03-02
2183   US0003071083    AAC  2018-01-03  2021-07-24
22831  KYG330321061    AAC  2021-10-24  2023-11-01
17951  US04316G2049   AACQ  2020-07-17  2020-07-17
18271  KY04316G1057   AACQ  2020-09-03  2021-06-25
26017  KYG330331128   AACT  2023-04-26  2023-06-12
26369  KYG330331045   AACT  2023-08-21  2025-06-24
21736  US0240031056   AADI  2021-08-27  2021-09-14
21985  US00032Q1040   AADI  2021-09-20  2025-06-19
1347   US02376R1023    AAL  2018-01-02  2025-06-30
5476   CA00782P1080    AAL  2018-01-17  2020-04-14
27496  KYG1000R1193    AAM  202

In [84]:
dups = df[df.duplicated(subset='Ticker', keep=False)].sort_values(['Ticker','Start_Date'])
print(dups.head(20))

              ISIN Ticker  Start_Date    End_Date   min_date   max_date  \
631   US00846U1016      A  2018-01-03  2025-06-24 2018-01-02 2024-03-28   
2582  CA04226J1084      A  2022-08-22  2022-10-04 2018-01-02 2024-03-28   
1555  US0138721065     AA  2018-01-13  2025-06-24 2019-01-02 2024-03-28   
2292  CA0120271089     AA  2020-01-28  2020-01-28 2019-01-02 2024-03-28   
481   US02376R1023    AAL  2018-01-02  2025-06-30 2018-01-02 2024-03-28   
1627  CA00782P1080    AAL  2018-01-17  2020-04-14 2018-01-02 2024-03-28   
1846  US0025353006    AAN  2018-01-22  2021-09-07 2018-01-02 2024-01-31   
2252  CA0496AP1036    AAN  2019-12-17  2021-04-26 2018-01-02 2024-01-31   
2495  US00258W1080    AAN  2021-10-07  2024-10-04 2018-01-02 2024-01-31   
2510  CA0496AP2026    AAN  2021-10-27  2024-05-22 2018-01-02 2024-01-31   
1806  US00751Y1064    AAP  2018-01-21  2025-06-30 2018-01-02 2024-03-28   
2213  CA02078D1042    AAP  2019-07-11  2021-04-16 2018-01-02 2024-03-28   
1378  US0240131047    AAT

In [86]:
dup_counts = df_tickers_us[df_tickers_us.duplicated(subset='Ticker', keep=False)].groupby('Ticker').size()
print(dup_counts.value_counts())
print(f"Total tickers with duplicates: {len(dup_counts)}")

2    1417
3     101
4      10
Name: count, dtype: int64
Total tickers with duplicates: 1528


In [87]:
df_tickers_us = df_tickers_us.sort_values('End_Date', ascending=False).drop_duplicates(subset='Ticker', keep='first')

print(df_tickers_us.shape)
print(df_tickers_us.duplicated(subset='Ticker').sum())

(16679, 4)
0


In [88]:
df = df_tickers_us.merge(date_ranges, on='Ticker', how='inner')
print(df.shape)

(1932, 7)


In [ ]:
df.to_csv()

In [89]:
print(df['Sector'].value_counts())

Sector
Financials                    363
Consumer Discretionary        332
Industrials                   294
Information Technology        253
Health Care                   233
Energy                        107
Materials                     101
Consumer Staples               86
Utilities                      71
Real Estate                    56
Telecommunication Services     23
Communication Services         13
Name: count, dtype: int64


In [90]:
df['Sector'] = df['Sector'].replace('Telecommunication Services', 'Communication Services')
print(df['Sector'].value_counts())

Sector
Financials                363
Consumer Discretionary    332
Industrials               294
Information Technology    253
Health Care               233
Energy                    107
Materials                 101
Consumer Staples           86
Utilities                  71
Real Estate                56
Communication Services     36
Name: count, dtype: int64


In [92]:
df.shape

(1932, 7)

In [96]:
df.to_csv(r'P:\Personal\Birkbeck\MSc Project\git_msc_project_a\data\processed\us_tickers_1.csv', index=False)

In [41]:
myfilename =r"T:\CGATES\Alexandria\US_tickerlist2.csv"
df_sec = pd.read_csv(myfilename, sep=',', header=0)

print(df_sec.columns.tolist())
print(df_sec.head(2))
print(df_sec.shape)

['bb_tcm', 'Ticker', 'SECURITY_TYP', 'Unnamed: 3', 'Unnamed: 4', 'Sec_Type_Summary', 'Proceed', 'Count']
            bb_tcm Ticker               SECURITY_TYP  Unnamed: 3  Unnamed: 4  \
0  DKNGW US Equity  DKNGW  #N/A Field Not Applicable         NaN         NaN   
1  ABBCL US Equity  ABBCL      #N/A Invalid Security         NaN         NaN   

            Sec_Type_Summary Proceed  Count  
0  #N/A Field Not Applicable      NO    1.0  
1      #N/A Invalid Security      NO  285.0  
(14900, 8)


In [9]:
df_tickers = df_tickers.merge(
    df_sec[['Ticker', 'SECURITY_TYP']].drop_duplicates(),
    on='Ticker',
    how='left')



In [23]:
df_sec.drop(columns=['Unnamed: 3','Unnamed: 4','Sec_Type_Summary','Proceed','Count'], inplace=True)

In [26]:
df_sec['SECURITY_TYP'].value_counts()

SECURITY_TYP
Common Stock                 10084
ETP                           1799
ADR                           1374
Closed-End Fund                607
REIT                           321
#N/A Invalid Security          285
Unit                           177
MLP                            111
Royalty Trst                    34
GDR                             28
Open-End Fund                   18
Ltd Part                        15
NY Reg Shrs                     10
Tracking Stk                    10
CDI                              6
Stapled Security                 3
Private Comp                     3
Misc.                            3
Receipt                          3
Preference                       2
Foreign Sh.                      1
Equity Index                     1
Dutch Cert                       1
#N/A Field Not Applicable        1
Equity WRT                       1
Index                            1
Savings Share                    1
Name: count, dtype: int64

In [28]:
df_sec.shape

(14900, 3)

In [16]:
import numpy as np

mask = (df_tickers['SECURITY_TYP'] == '#N/A Invalid Security') | (df_tickers['SECURITY_TYP'].isna())

invalid_isins = df_tickers.loc[mask, 'ISIN']
print(invalid_isins)
print(f"Count: {len(invalid_isins)}")

invalid_isins.to_clipboard()

1        US0846701086
2        US0846707026
7        CA0213611001
9        CA88581L1058
10       CA47215Q1046
             ...     
29303    US33740F1104
29304    US80590A1051
29305    US71913T1034
29306    KYG4791J1224
29307    KYG2254C1050
Name: ISIN, Length: 12151, dtype: object
Count: 12151


In [11]:
df_tickers[['Ticker','ISIN']].to_clipboard()

In [13]:
df_tickers.shape

(29308, 6)

In [14]:
df_tickers.head(50)

,ISIN,Ticker,Start_Date,End_Date,Sector,SECURITY_TYP
0,US0970231058,BA,2018-01-01,2025-06-30,Industrials,Common Stock
1,US0846701086,BRKA,2018-01-01,2025-06-30,NaN,#N/A Invalid Security
2,US0846707026,BRKB,2018-01-01,2025-06-30,Financials,#N/A Invalid Security
3,US12572Q1058,CME,2018-01-01,2025-06-30,Financials,Common Stock
4,US7081601061,JCP,2018-01-01,2021-01-06,Consumer Discretionary,Common Stock
5,US55616P1049,M,2018-01-01,2025-06-07,Consumer Discretionary,Common Stock
6,US8123501061,SHLD,2018-01-01,2024-10-01,Consumer Discretionary,ETP
7,CA0213611001,ALA,2018-01-01,2025-06-27,NaN,NaN
8,US7168301049,PUGOY,2018-01-01,2021-01-20,NaN,ADR
9,CA88581L1058,IDK,2018-01-01,2019-11-22,NaN,NaN


In [20]:
myfilename =r"T:\CGATES\Alexandria\US_tickerlist3.csv"
dfs2 = pd.read_csv(myfilename, sep=',', header=0)

print(dfs2.columns.tolist())
print(dfs2.head(2))

['Ticker', 'ISIN', 'TICKER_AND_EXCH_CODE', 'Unnamed: 3', 'Unnamed: 4']
  Ticker          ISIN       TICKER_AND_EXCH_CODE  Unnamed: 3 Unnamed: 4
0  HBANP  US4461504015  #N/A Field Not Applicable         NaN        NaN
1  CHSCP  US12542R2094  #N/A Field Not Applicable         NaN        NaN


In [22]:
dfs2.shape

(29308, 5)

In [35]:
myfilename =r"T:\CGATES\Alexandria\060726_bqnt_pull_1.txt"
df1 = pd.read_csv(myfilename, sep='\t', header=0)

print(df1.columns.tolist())
print(df1.head(2))
print(df1.shape)

['Unnamed: 0', 'bb_tcm', 'isin', 'start_date', 'end_date', 'average_last_price', 'average_volume', 'average_traded_volume', 'average_market_cap', 'ln_mktcap', 'ln_vol', 'ln_val', 'atv_M']
   Unnamed: 0              bb_tcm          isin  start_date    end_date  \
0           1  1573402D US Equity  US40053C1053  2018-01-18  2018-05-29   
1           2  1624579D US Equity  US83408W1036  2018-01-15  2019-11-05   

   average_last_price  average_volume  average_traded_volume  \
0            7.824286    1.906608e+06           1.488416e+07   
1           35.719368    5.235073e+05           1.850647e+07   

   average_market_cap  ln_mktcap     ln_vol     ln_val      atv_M  
0           32.128372   3.500390  14.460837  16.515808  14.884161  
1            1.389211   0.870963  13.168308  16.733631  18.506467  
(2895, 13)


In [36]:
df1.shape

(2895, 13)

In [39]:
df1['isin'].nunique()


2895

In [40]:
missing = set(df1['isin']) - set(df_tickers['ISIN'])
print(f"{len(missing)} ISINs in B not found in A")
print(missing)

0 ISINs in B not found in A
set()


In [42]:
df1 = df1.rename(columns={'isin': 'ISIN'})

In [43]:
df_tickers = df_tickers.merge(
    df1.drop_duplicates(),
    on='ISIN',
    how='left')

In [44]:
print(df_tickers.shape)
print(df_tickers.isnull().sum())

(29308, 18)
ISIN                         0
Ticker                       2
Start_Date                   0
End_Date                     0
Sector                   25240
SECURITY_TYP             11983
Unnamed: 0               26191
bb_tcm                   26191
start_date               26191
end_date                 26191
average_last_price       26191
average_volume           26191
average_traded_volume    26191
average_market_cap       26198
ln_mktcap                26198
ln_vol                   26191
ln_val                   26191
atv_M                    26191
dtype: int64


In [59]:
df_tickers[df_tickers['SECURITY_TYP'].isna()]

,ISIN,Ticker,Start_Date,End_Date,Sector,SECURITY_TYP,Unnamed: 0,bb_tcm,start_date,end_date,average_last_price,average_volume,average_traded_volume,average_market_cap,ln_mktcap,ln_vol,ln_val,atv_M
7,CA0213611001,ALA,2018-01-01,2025-06-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,CA88581L1058,IDK,2018-01-01,2019-11-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,CA47215Q1046,PJC.A,2018-01-01,2018-05-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,CA3197021064,FCC,2018-01-01,2021-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
361,CA0977511017,BBD.A,2018-01-02,2025-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29303,US33740F1104,FBDC,2025-06-30,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29304,US80590A1051,SCAG,2025-06-30,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29305,US71913T1034,PXPCD,2025-06-30,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29306,KYG4791J1224,INACU,2025-06-30,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
mask = ((df_tickers['SECURITY_TYP'].isna()) & ( df_tickers['bb_tcm'].isna()))
x = df_tickers.loc[mask]




In [61]:
x.to_csv(r'T:\CGATES\Alexandria\bb_tcm_finding_3.csv', index=False)

In [55]:
myfilename =r"T:\CGATES\Alexandria\bb_tcm_finding_2.csv"
df2 = pd.read_csv(myfilename, sep=',', header=0)
print(df2.columns)
print(df2.shape)

Index(['Unnamed: 0.1', 'Unnamed: 1', 'ISIN', 'Ticker', 'Start_Date',
       'End_Date', 'Sector', 'SECURITY_TYP', 'Unnamed: 0', 'bb_tcm',
       'ULT_PARENT_TICKER_EXCHANGE', 'INDUSTRY_SECTOR'],
      dtype='object')
(149, 12)


In [57]:
df2.drop(columns=['Unnamed: 0.1','Unnamed: 1','Unnamed: 0'], inplace=True)

In [58]:
df2[df2['']]

,ISIN,Ticker,Start_Date,End_Date,Sector,SECURITY_TYP,bb_tcm,ULT_PARENT_TICKER_EXCHANGE,INDUSTRY_SECTOR
0,US3456051099,FCEA,01/01/2018,11/12/2018,NaN,#N/A Invalid Security,FCE/A US,BN CN,Financial
1,US0549371070,BBT,02/01/2018,13/12/2019,Financials,#N/A Invalid Security,NaN,#N/A Invalid Security,#N/A Invalid Security
2,US97382A2006,WIN,02/01/2018,15/03/2019,Consumer Staples,#N/A Invalid Security,B4O QT,UNIT US,Communications
3,US9426221019,WSOB,02/01/2018,06/10/2022,NaN,#N/A Invalid Security,WSO/B US,WSO US,"Consumer, Cyclical"
4,US46122T1025,XON,02/01/2018,04/02/2020,NaN,#N/A Invalid Security,NaN,#N/A Invalid Security,#N/A Invalid Security
5,US3976242061,GEFB,02/01/2018,11/06/2025,NaN,#N/A Invalid Security,GEF/B US,GEF US,Industrial
6,US2246332066,CRDA,02/01/2018,30/06/2025,NaN,#N/A Invalid Security,CRD/A US,CRD/A US,Financial
7,US2246331076,CRDB,02/01/2018,30/06/2025,NaN,#N/A Invalid Security,CRD/B US,CRD/A US,Financial
8,US09248J1016,BNJ,02/01/2018,25/05/2018,NaN,#N/A Invalid Security,2211783D US,BLK US,Funds
9,US16948W1009,HGSH,03/01/2018,16/08/2021,NaN,#N/A Invalid Security,NaN,#N/A Invalid Security,#N/A Invalid Security


In [98]:
df_tickers[df_tickers['ISIN']=='US36472T1097']

,ISIN,Ticker,Start_Date,End_Date
16246,US36472T1097,GCI,2020-01-02,2025-06-24
